# Sensitivity to cell-type label granularity and to the number of empty barcodes (PBMC 8k)

CellSweep fits one expression profile per input label, so the granularity of the labels sets what it treats as shared
cell-type signal. This notebook reruns CellSweep on the 8,381 PBMC 8k cells with labelings from a single label up to
Leiden resolution 20 (190 clusters), the CellTypist Immune_All_High (used elsewhere in the paper) and Immune_All_Low
annotations, and a shuffled-label control; with 10 to all 728,899 non-cellular barcodes; and on simulated data with
known noise. All runs use the raw matrix and empty-droplet calls of the default PBMC 8k CellSweep run
(`benchmarking.ipynb`). The last section draws the paper's Fig. S19.

In [1]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors

import cellsweep.utils as cs_utils
from cellsweep import denoise_count_matrix

cellsweep_dir = os.path.dirname(os.path.abspath(""))
threads = 16
overwrite = False   # rerun labelings and CellSweep runs that already exist

## 1. Labelings (PBMC 8k)

`all_one` (one label), `lineage` (lymphoid/myeloid), `ct_high` (CellTypist Immune_All_High, the default), `ct_low`
(Immune_All_Low), Leiden at resolutions 0.001 to 20 with the same scanpy pipeline as `benchmarking.ipynb`, and
`shuffled` (`ct_high` permuted across cells). Written to `labels.csv` once.

In [2]:
dataset_name = "pbmc8k"
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", dataset_name)
run_dir = os.path.join(data_dir, "celltype_granularity")
os.makedirs(run_dir, exist_ok=True)
default_cellsweep_path = os.path.join(data_dir, "pbmc8k_output_cellsweep.h5ad")
labels_csv = os.path.join(run_dir, "labels.csv")
min_genes = 0
min_cells = 0
max_mt_percentage = None
n_top_genes = 2000
n_pcs = 25
n_neighbors = 20
expected_cells = 8381
cellsweep_max_iter = 2000
cellsweep_init_alpha = 0.9
cellsweep_init_beta = 0.01
existing_leiden_resolutions = [0.1, 0.5, 1.0, 1.5, 2.0, 5.0]
new_leiden_resolutions = [0.001, 0.005, 0.01, 10.0, 20.0]  # 0.001 -> K=1, 0.005 -> K=2, 0.01 -> K=3
LINEAGE = {
    "T cells": "Lymphoid", "ILC": "Lymphoid", "B cells": "Lymphoid",
    "Monocytes": "Myeloid", "DC": "Myeloid", "pDC": "Myeloid", "HSC/MPP": "Myeloid",
}
def res_tag(r):
    return str(float(r)).replace(".", "_")
def load_raw():
    """Raw counts + empty-droplet calls + CellTypist-High labels, from the default run."""
    adata = ad.read_h5ad(default_cellsweep_path)
    adata.X = adata.layers["raw"].copy()
    del adata.layers["raw"]
    adata.obs = adata.obs[["is_empty", "celltype"]].copy()
    for key in list(adata.uns.keys()):
        del adata.uns[key]
    adata.var = adata.var[["gene_ids", "empty_counts"]].copy()
    adata.var_names_make_unique()
    return adata
def make_labels(overwrite=False):
    if os.path.exists(labels_csv) and not overwrite:
        return pd.read_csv(labels_csv, index_col=0, dtype=str)

    adata = load_raw()
    cells = adata[~adata.obs["is_empty"]].copy()
    labels = pd.DataFrame(index=cells.obs_names)

    labels["ct_high"] = cells.obs["celltype"].astype(str)
    labels["all_one"] = "cell"
    labels["lineage"] = labels["ct_high"].map(LINEAGE)
    assert labels["lineage"].notna().all(), labels.loc[labels["lineage"].isna(), "ct_high"].unique()

    rng = np.random.default_rng(0)
    labels["shuffled"] = rng.permutation(labels["ct_high"].values)

    # CellTypist Immune_All_Low (same call the notebook uses for High)
    ct = cs_utils.determine_cell_types(adata, model_pkl="Immune_All_Low.pkl", filter_empty=True, expected_cells=expected_cells, verbose=1)
    labels["ct_low"] = ct.obs["celltype"].astype(str).reindex(labels.index)

    # Leiden labels already used for the notebook sweep
    for r in existing_leiden_resolutions:
        path = os.path.join(data_dir, f"adata_cellsweep_leiden_{res_tag(r)}.h5ad")
        obs = ad.read_h5ad(path, backed="r").obs
        labels[f"leiden_{r:g}"] = obs["celltype"].astype(str).reindex(labels.index)

    # finer Leiden resolutions, same pipeline as the notebook
    for r in new_leiden_resolutions:
        tmp = cs_utils.run_scanpy_preprocessing_and_clustering(cells, min_genes=min_genes, min_cells=min_cells, max_mt_percentage=max_mt_percentage, n_top_genes=n_top_genes, n_pcs=n_pcs, n_neighbors=n_neighbors, leiden_resolution=r, seed=42, verbose=1)
        labels[f"leiden_{r:g}"] = tmp.obs["leiden"].astype(str).reindex(labels.index)

    labels = labels[[c for c in labels.columns if not c.startswith("leiden")] + sorted([c for c in labels.columns if c.startswith("leiden")], key=lambda c: float(c.split("_")[1]))]
    assert not labels.isna().any().any(), labels.isna().sum()
    labels.to_csv(labels_csv)
    return labels
def run_one(condition, threads, overwrite=False):
    out = os.path.join(run_dir, f"adata_cellsweep_{condition}.h5ad")
    if os.path.exists(out) and not overwrite:
        return condition, "exists"
    labels = pd.read_csv(labels_csv, index_col=0, dtype=str)
    adata = load_raw()
    adata.obs["celltype"] = pd.Categorical(labels[condition].reindex(adata.obs_names).fillna("Empty Droplet"))
    cs_utils_log = os.path.join(run_dir, f"cellsweep_{condition}.log")
    from cellsweep import denoise_count_matrix
    denoise_count_matrix(adata, adata_out=out, init_alpha=cellsweep_init_alpha, init_beta=cellsweep_init_beta, freeze_ambient_profile=True, max_iter=cellsweep_max_iter, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=1, log_file=cs_utils_log)
    return condition, "done"

labels = make_labels(overwrite=overwrite)
labels.nunique()

ct_high           7
all_one           1
lineage           2
shuffled          7
ct_low           18
leiden_0.001      2
leiden_0.005      3
leiden_0.01       3
leiden_0.1        6
leiden_0.5       14
leiden_1         19
leiden_1.5       23
leiden_2         25
leiden_5         48
leiden_10        95
leiden_20       202
dtype: int64

## 2. CellSweep on each labeling

In [3]:
for condition in labels.columns:
    print(condition, run_one(condition, threads, overwrite=overwrite)[1], flush=True)

ct_high exists


all_one exists


lineage exists


shuffled exists


ct_low exists


...

leiden_1.5 exists


leiden_2 exists


leiden_5 exists


leiden_10 exists


leiden_20 exists


## 3. Number of empty barcodes

Same cells and `ct_high` labels; the non-cellular barcodes are subsampled (seed 42) to 10 to 500,000, or all are used.

In [3]:
empty_run_dir = os.path.join(data_dir, "empty_barcode_sweep")
os.makedirs(empty_run_dir, exist_ok=True)
EMPTY_BARCODE_NUMBERS = [10, 100, 1_000, 10_000, 50_000, 100_000, 500_000, None]  # None = all of them
SEED = 42
def empty_run_one(n_empty, threads, overwrite=False):
    tag = "all" if n_empty is None else str(n_empty)
    out = os.path.join(empty_run_dir, f"adata_cellsweep_empty_{tag}.h5ad")
    if os.path.exists(out) and not overwrite:
        return tag, "exists"
    from cellsweep import denoise_count_matrix
    labels = pd.read_csv(labels_csv, index_col=0, dtype=str)
    adata = load_raw()
    adata.obs["celltype"] = pd.Categorical(labels["ct_high"].reindex(adata.obs_names).fillna("Empty Droplet"))
    if n_empty is not None:
        empty = adata.obs_names[adata.obs["is_empty"].values]
        rng = np.random.default_rng(SEED)
        keep = set(rng.choice(empty, size=n_empty, replace=False))
        adata = adata[~adata.obs["is_empty"].values | adata.obs_names.isin(keep)].copy()
    denoise_count_matrix(adata, adata_out=out, init_alpha=cellsweep_init_alpha, init_beta=cellsweep_init_beta, freeze_ambient_profile=True, max_iter=cellsweep_max_iter, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=1, log_file=os.path.join(empty_run_dir, f"cellsweep_empty_{tag}.log"))
    return tag, "done"

for n_empty in EMPTY_BARCODE_NUMBERS:
    print(n_empty, empty_run_one(n_empty, threads, overwrite=overwrite)[1], flush=True)

10 exists


100 exists


1000 exists


10000 exists


50000 exists


100000 exists


500000 exists


None exists


## 4. Simulation with ground truth

`simulation1_small_noise` (12 true cell types, known real and noise counts per entry): true types pooled into K groups,
the true labels, random splits of each type, Leiden labels, and shuffled true labels.

In [5]:
sim_dataset_name = "simulation1_small_noise"
sim_data_dir = os.path.join(cellsweep_dir, "notebooks", "data", sim_dataset_name)
sim_out_dir = os.path.join(cellsweep_dir, "notebooks", "output", sim_dataset_name)
sim_run_dir = os.path.join(sim_data_dir, "celltype_granularity")
os.makedirs(sim_run_dir, exist_ok=True)
os.makedirs(sim_out_dir, exist_ok=True)
sim_raw_path = os.path.join(sim_data_dir, "adata_raw.h5ad")
sim_labels_csv = os.path.join(sim_run_dir, "labels.csv")
sim_metrics_csv = os.path.join(sim_out_dir, "celltype_granularity_simulation_metrics.csv")
sim_expected_cells = 10000
sim_cellsweep_max_iter = 2000
def sim_make_labels(overwrite=False):
    if os.path.exists(sim_labels_csv) and not overwrite:
        return pd.read_csv(sim_labels_csv, index_col=0, dtype=str)
    adata = ad.read_h5ad(sim_raw_path)
    cells = adata[~adata.obs["is_empty"]].copy()
    truth = cells.obs["celltype"].astype(str)
    type_idx = truth.str.replace("Type_", "").astype(int).values
    rng = np.random.default_rng(0)

    labels = pd.DataFrame(index=cells.obs_names)
    # pool true types into K groups with a fixed random assignment
    perm = rng.permutation(12)
    for K in [1, 2, 3, 6]:
        group_of_type = {t: int(np.where(perm == t)[0][0]) % K for t in range(12)}
        labels[f"merge_{K}"] = [f"G{group_of_type[t]}" for t in type_idx]
    labels["truth"] = truth.values
    for m in [2, 4, 8]:
        labels[f"split_{m}"] = [f"{t}_{s}" for t, s in zip(truth.values, rng.integers(0, m, size=len(truth)))]
    for r in [1.0, 5.0, 20.0]:
        tmp = cs_utils.run_scanpy_preprocessing_and_clustering(cells, min_genes=0, min_cells=0, max_mt_percentage=None, n_top_genes=2000, n_pcs=25, n_neighbors=20, leiden_resolution=r, seed=42, verbose=0)
        labels[f"leiden_{r:g}"] = tmp.obs["leiden"].astype(str).reindex(labels.index)
    labels["shuffled"] = rng.permutation(truth.values)
    assert not labels.isna().any().any()
    labels.to_csv(sim_labels_csv)
    return labels
def sim_run_one(condition, threads, overwrite=False):
    out = os.path.join(sim_run_dir, f"adata_cellsweep_{condition}.h5ad")
    if os.path.exists(out) and not overwrite:
        return condition, "exists"
    from cellsweep import denoise_count_matrix
    labels = pd.read_csv(sim_labels_csv, index_col=0, dtype=str)
    adata = ad.read_h5ad(sim_raw_path)
    adata.obs["celltype"] = pd.Categorical(labels[condition].reindex(adata.obs_names).fillna("Empty Droplet"))
    denoise_count_matrix(adata, adata_out=out, init_alpha=0.9, init_beta=0.01, freeze_ambient_profile=True, max_iter=sim_cellsweep_max_iter, empty_droplet_method="threshold", sim_expected_cells=sim_expected_cells, threads=threads, verbose=1, log_file=os.path.join(sim_run_dir, f"cellsweep_{condition}.log"))
    return condition, "done"
def sim_score(condition, labels):
    a = ad.read_h5ad(os.path.join(sim_run_dir, f"adata_cellsweep_{condition}.h5ad"))
    a = a[~a.obs["is_empty"]].copy()
    raw = sp.csr_matrix(a.layers["raw"]).astype(np.float64)
    den = sp.csr_matrix(a.X).astype(np.float64)
    real = sp.csr_matrix(a.layers["real"]).astype(np.float64)
    noise = sp.csr_matrix(a.layers["noise"]).astype(np.float64)
    removed = raw - den
    removed.data = np.clip(removed.data, 0, None)

    correctly_removed = removed.minimum(noise).sum()
    real_retained = den.minimum(real).sum()

    # per-entry error on entries that are nonzero in raw
    err = (den - real)
    # per-cell contamination fraction
    raw_tot = np.asarray(raw.sum(1)).ravel()
    est_frac = np.asarray(removed.sum(1)).ravel() / raw_tot
    true_frac = np.asarray(noise.sum(1)).ravel() / raw_tot

    # off-target markers: markers of type t counted in cells whose TRUE type is not t
    marker_sets = a.uns["marker_sets"]
    truth = labels["truth"].reindex(a.obs_names).values
    off_raw = off_rem = on_raw = on_ret = 0.0
    raw_c, den_c = raw.tocsc(), den.tocsc()
    for t, genes in enumerate(marker_sets):
        genes = np.asarray(genes).astype(int)
        on_mask = truth == f"Type_{t}"
        r = raw_c[:, genes]
        d = den_c[:, genes]
        off_raw += r[~on_mask].sum()
        off_rem += (r[~on_mask] - d[~on_mask]).sum()
        on_raw += r[on_mask].sum()
        on_ret += d[on_mask].sum()

    return dict(
        condition=condition,
        n_labels=labels[condition].nunique(),
        noise_recall=correctly_removed / noise.sum(),
        removal_precision=correctly_removed / removed.sum(),
        real_retained=real_retained / real.sum(),
        fraction_removed=removed.sum() / raw.sum(),
        true_fraction_noise=noise.sum() / raw.sum(),
        entry_rmse=np.sqrt(err.multiply(err).sum() / raw.nnz),
        cell_frac_spearman=spearmanr(est_frac, true_frac).correlation,
        cell_frac_mae=np.mean(np.abs(est_frac - true_frac)),
        offtarget_marker_removed=off_rem / off_raw,
        ontarget_marker_retained=on_ret / on_raw,
        mean_cell_size_per_label=len(labels) / labels[condition].nunique(),
    )

sim_labels = sim_make_labels(overwrite=overwrite)
for condition in sim_labels.columns:
    print(condition, sim_run_one(condition, threads, overwrite=overwrite)[1], flush=True)
sim_metrics = pd.DataFrame([sim_score(c, sim_labels) for c in sim_labels.columns])
sim_metrics.to_csv(sim_metrics_csv, index=False)
sim_metrics.round(4)

merge_1 exists


merge_2 exists


merge_3 exists


merge_6 exists


truth exists


...

leiden_1 exists


leiden_5 exists


leiden_20 exists


shuffled exists


,condition,n_labels,noise_recall,removal_precision,real_retained,fraction_removed,true_fraction_noise,entry_rmse,cell_frac_spearman,cell_frac_mae,offtarget_marker_removed,ontarget_marker_retained,mean_cell_size_per_label
0,merge_1,1,0.8118,0.0466,0.3101,0.6948,0.0399,8.9305,0.0551,0.6577,0.9441,0.3083,10157.0000
1,merge_2,2,0.5427,0.0750,0.7222,0.2884,0.0399,5.2929,0.4787,0.2535,0.7936,0.7370,5078.5000
2,merge_3,3,0.4381,0.1710,0.9118,0.1021,0.0399,2.5361,0.7787,0.0682,0.7828,0.9267,3385.6667
3,merge_6,6,0.4347,0.4288,0.9760,0.0404,0.0399,0.3715,0.9113,0.0068,0.9059,0.9912,1692.8333
4,truth,12,0.4571,0.4555,0.9773,0.0400,0.0399,0.3586,0.9230,0.0062,0.9946,0.9945,846.4167
5,split_2,24,0.4532,0.4555,0.9775,0.0397,0.0399,0.3589,0.9195,0.0064,0.9870,0.9946,423.2083
6,split_4,48,0.4468,0.4553,0.9778,0.0391,0.0399,0.3595,0.9125,0.0066,0.9737,0.9946,211.6042
7,split_8,96,0.4324,0.4551,0.9785,0.0379,0.0399,0.3607,0.9015,0.0070,0.9424,0.9947,105.8021
8,leiden_1,12,0.4571,0.4555,0.9773,0.0400,0.0399,0.3586,0.9230,0.0062,0.9946,0.9945,846.4167
9,leiden_5,32,0.4531,0.4545,0.9774,0.0397,0.0399,0.3588,0.9243,0.0064,0.9870,0.9946,317.4062


## 5. Metrics

Scored with a fixed reference annotation (CellTypist Immune_All_Low collapsed to major populations): lineage-marker
removal from expressing and non-expressing cells, subtype markers inside a coarse label, per-cell and per-gene removal
fractions, and kNN purity of the reference populations and of each run's own labels.

In [4]:
matplotlib.use("Agg")
gran_out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "pbmc8k", "celltype_granularity")
os.makedirs(gran_out_dir, exist_ok=True)
DEFAULT = "ct_high"
REF_GROUP = {
    "Tcm/Naive helper T cells": "CD4 T", "Tem/Effector helper T cells": "CD4 T", "Regulatory T cells": "CD4 T", "Treg(diff)": "CD4 T",
    "Tcm/Naive cytotoxic T cells": "CD8 T", "Tem/Temra cytotoxic T cells": "CD8 T", "Tem/Trm cytotoxic T cells": "CD8 T",
    "MAIT cells": "MAIT",
    "CD16+ NK cells": "NK", "NK cells": "NK",
    "Naive B cells": "Naive B", "Memory B cells": "Memory B",
    "Classical monocytes": "CD14 Mono", "Non-classical monocytes": "CD16 Mono",
    "DC1": "cDC", "DC2": "cDC", "pDC": "pDC",
    "Megakaryocytes/platelets": "Platelet", "HSC/MPP": "HSC",
}
T = {"CD4 T", "CD8 T", "MAIT"}
B = {"Naive B", "Memory B"}
MONO = {"CD14 Mono", "CD16 Mono"}
MYELOID = MONO | {"cDC", "pDC"}
ALL = set(REF_GROUP.values())
LINEAGE_PANELS = {
    "Myeloid (LYZ, S100A8/9, CST3, FCN1)": (["LYZ", "S100A8", "S100A9", "CST3", "FCN1"], MYELOID, T | B | {"NK"}),
    "B (MS4A1, CD79A/B)": (["MS4A1", "CD79A", "CD79B"], B, T | MONO | {"NK"}),
    "T (CD3D/E, TRAC)": (["CD3D", "CD3E", "TRAC"], T, B | MYELOID),
    "NK (NKG7, GNLY, KLRF1)": (["NKG7", "GNLY", "KLRF1"], {"NK"}, B | MYELOID),
    "Platelet (PPBP, PF4)": (["PPBP", "PF4"], {"Platelet"}, ALL - {"Platelet"}),
    "Erythrocyte (HBB, HBA1/2)": (["HBB", "HBA1", "HBA2"], set(), ALL),
}
SUBTYPE_MARKERS = {
    "CD8A": ("CD8 T", "CD4 T"), "CD8B": ("CD8 T", "CD4 T"),
    "SLC4A10": ("MAIT", "CD4 T"), "KLRB1": ("MAIT", "CD4 T"),
    "FCGR3A": ("CD16 Mono", "CD14 Mono"), "CDKN1C": ("CD16 Mono", "CD14 Mono"), "MS4A7": ("CD16 Mono", "CD14 Mono"),
    "CD14": ("CD14 Mono", "CD16 Mono"), "VCAN": ("CD14 Mono", "CD16 Mono"), "S100A12": ("CD14 Mono", "CD16 Mono"),
    "TCL1A": ("Naive B", "Memory B"), "IGHD": ("Naive B", "Memory B"),
}
N_NEIGHBORS = 15
N_PCS = 25
def condition_order():
    k = labels.nunique()
    order = sorted([c for c in labels.columns if c != "shuffled"], key=lambda c: (k[c], c))
    return order + ["shuffled"]
def pretty(c):
    names = {"all_one": "Single label", "lineage": "Lymphoid/myeloid", "ct_high": "CellTypist High (default)", "ct_low": "CellTypist Low", "shuffled": "Shuffled High (control)"}
    if c in names:
        return f"{names[c]} (K={labels[c].nunique()})"
    return f"Leiden r={c.split('_')[1]} (K={labels[c].nunique()})"
def load(condition, genes):
    a = ad.read_h5ad(os.path.join(run_dir, f"adata_cellsweep_{condition}.h5ad"))
    a = a[~a.obs["is_empty"].values].copy()
    a.var_names_make_unique()
    a = a[labels.index]
    raw = sp.csr_matrix(a.layers["raw"], dtype=np.float64)
    den = sp.csr_matrix(a.X, dtype=np.float64)
    return a, raw, den
def knn_purity(emb, groups):
    nn = NearestNeighbors(n_neighbors=N_NEIGHBORS + 1).fit(emb)
    idx = nn.kneighbors(emb, return_distance=False)[:, 1:]
    g = np.asarray(groups)
    return float(np.mean(g[idx] == g[:, None]))
def embed(X, hvg_idx):
    ad_tmp = ad.AnnData(X=X[:, hvg_idx].copy())
    sc.pp.normalize_total(ad_tmp, target_sum=1e4)
    sc.pp.log1p(ad_tmp)
    sc.pp.scale(ad_tmp, max_value=10)
    sc.tl.pca(ad_tmp, n_comps=N_PCS, random_state=0)
    return ad_tmp.obsm["X_pca"]
def make_figure(summary, lineage, subtype, order):
    plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
    s = summary.set_index("condition").loc[order]
    main_conds = [c for c in order if c != "shuffled"]
    ms = s.loc[main_conds]
    x = ms["n_labels"].values
    named = {"all_one": "1 label", "lineage": "lineage", "ct_high": "CT High", "ct_low": "CT Low"}
    ct_color, leiden_color, coarse_color, shuf_color = "#DD8452", "#4C72B0", "#8C8C8C", "#C44E52"

    def ccolor(c):
        return leiden_color if c.startswith("leiden") else ct_color if c.startswith("ct_") else coarse_color

    def points(ax, series, label_points=True, shuffled_value=None):
        leiden = [c for c in main_conds if c.startswith("leiden")]
        ax.plot(s.loc[leiden, "n_labels"], series.loc[leiden], "-", color=leiden_color, lw=1, zorder=2)
        for c in main_conds:
            ax.scatter(s.loc[c, "n_labels"], series.loc[c], color=ccolor(c), s=20, zorder=3)
            if label_points and c in named:
                ax.annotate(named[c], (s.loc[c, "n_labels"], series.loc[c]), textcoords="offset points", xytext=(4, 3), fontsize=6)
        if shuffled_value is not None:
            ax.scatter(s.loc["shuffled", "n_labels"], shuffled_value, marker="x", color=shuf_color, s=28, zorder=4)
        ax.set_xscale("log")
        ax.set_xlabel("Number of labels K (log)")

    fig, axes = plt.subplots(2, 3, figsize=(11, 7))

    # a: pooled lineage-marker removal, off vs on target
    ax = axes[0, 0]
    points(ax, s["pooled_offtarget_marker_removed"], shuffled_value=s.loc["shuffled", "pooled_offtarget_marker_removed"])
    points(ax, s["pooled_ontarget_marker_removed"], label_points=False, shuffled_value=s.loc["shuffled", "pooled_ontarget_marker_removed"])
    ax.text(0.98, 0.62, "off-target counts\n(should be removed)", transform=ax.transAxes, ha="right", fontsize=6.5)
    ax.text(0.98, 0.12, "on-target counts\n(should be kept)", transform=ax.transAxes, ha="right", fontsize=6.5)
    ax.set_ylabel("Fraction of lineage-marker counts removed")
    ax.set_title("a  Lineage markers, pooled", loc="left", fontsize=8.5)

    # b / c: per panel on-target (over-correction) and off-target (under-correction)
    # line = merged-High labels then Leiden; squares = CellTypist High/Low; x = shuffled
    line_conds = ["all_one", "lineage"] + [c for c in main_conds if c.startswith("leiden")]
    panel_colors = dict(zip(LINEAGE_PANELS.keys(), plt.cm.tab10.colors))
    for ax, col, title, ylabel in [
        (axes[0, 1], "ontarget_removed", "b  Over-correction (on-target removed)", "Fraction removed from expressing cells"),
        (axes[0, 2], "offtarget_removed", "c  Under-correction (off-target removed)", "Fraction removed from non-expressing cells"),
    ]:
        for panel, sub in lineage.groupby("panel", sort=False):
            sub = sub.set_index("condition")
            if sub[col].isna().all():
                continue
            color = panel_colors[panel]
            ax.plot(s.loc[line_conds, "n_labels"], sub.loc[line_conds, col], "-o", ms=2.5, lw=1, color=color, label=panel)
            ax.scatter(s.loc[["ct_high", "ct_low"], "n_labels"], sub.loc[["ct_high", "ct_low"], col], marker="s", s=22, color=color, edgecolor="k", lw=0.5, zorder=4)
            ax.scatter(s.loc["shuffled", "n_labels"], sub.loc["shuffled", col], marker="x", color=color, s=22)
        ax.set_xscale("log")
        ax.set_xlabel("Number of labels K (log)")
        ax.set_ylabel(ylabel)
        ax.set_title(title, loc="left", fontsize=8.5)
    axes[0, 2].legend(fontsize=5.5, frameon=False, loc="lower left")
    axes[0, 1].text(0.97, 0.97, "line: merged High, then Leiden\nsquares: CellTypist High / Low\nx: shuffled High", transform=axes[0, 1].transAxes, ha="right", va="top", fontsize=6)

    # d: subtype-marker retention heatmap
    ax = axes[1, 0]
    piv = subtype.pivot(index="gene", columns="condition", values="retained_in_expressing")[order].loc[list(SUBTYPE_MARKERS.keys())]
    im = ax.imshow(piv.values, aspect="auto", cmap="viridis", vmin=0.8, vmax=1.0)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=4.5, color="w" if v < 0.9 else "k")
    ax.set_yticks(range(piv.shape[0]))
    ax.set_yticklabels([f"{g} in {SUBTYPE_MARKERS[g][0]}" for g in piv.index], fontsize=6)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([f"{named.get(c, c.replace('leiden_', 'r=')) if c != 'shuffled' else 'shuffled'} ({labels[c].nunique()})" for c in order], rotation=90, fontsize=6)
    cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cb.set_label("Fraction retained (clipped at 0.8)", fontsize=6)
    ax.set_title("d  Subtype markers in the expressing subtype", loc="left", fontsize=8.5)

    # e: model parameters absorb contamination as K grows
    ax = axes[1, 1]
    points(ax, s["fraction_counts_removed"], shuffled_value=s.loc["shuffled", "fraction_counts_removed"])
    ax2 = ax.twinx()
    leiden = [c for c in main_conds if c.startswith("leiden")] 
    ax2.plot(ms["n_labels"], ms["beta_hat"], ":", color="k", lw=1)
    ax2.scatter(ms["n_labels"], ms["beta_hat"], marker="d", s=10, color="k")
    ax2.set_ylabel("beta_hat (dotted)", fontsize=7)
    ax2.spines["right"].set_visible(True)
    ax.set_ylabel("Fraction of all counts removed")
    ax.set_title("e  Total removal and beta_hat", loc="left", fontsize=8.5)

    # f: circularity control
    ax = axes[1, 2]
    points(ax, s["own_label_purity_gain"], shuffled_value=s.loc["shuffled", "own_label_purity_gain"])
    ax.axhline(0, color="k", ls=":", lw=1)
    ax.set_ylim(-0.05, 0.05)
    ax.set_ylabel(f"{N_NEIGHBORS}-NN purity of input labels,\ndenoised minus raw")
    ax.set_title("f  Label self-reinforcement", loc="left", fontsize=8.5)

    handles = [plt.Line2D([], [], marker="o", ls="", color=ct_color, label="CellTypist"), plt.Line2D([], [], marker="o", ls="-", color=leiden_color, label="Leiden"),
               plt.Line2D([], [], marker="o", ls="", color=coarse_color, label="Merged CellTypist High"), plt.Line2D([], [], marker="x", ls="", color=shuf_color, label="Shuffled High (control)")]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, fontsize=7)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    for ext in ["png", "pdf"]:
        fig.savefig(os.path.join(gran_out_dir, f"celltype_granularity_sensitivity.{ext}"), dpi=300)
    plt.close(fig)

    if os.path.exists(sim_metrics_csv):
        sim = pd.read_csv(sim_metrics_csv)
        sim_main = sim[sim.condition != "shuffled"].sort_values("n_labels")
        shuf = sim[sim.condition == "shuffled"].iloc[0]
        styles = {"leiden": "o", "merge": "s", "split": "^", "truth": "*"}
        fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
        for ax, cols, ylab in [
            (axes[0], ["fraction_removed", "true_fraction_noise"], "Fraction of counts"),
            (axes[1], ["offtarget_marker_removed", "ontarget_marker_retained", "real_retained"], "Fraction"),
            (axes[2], ["cell_frac_spearman"], "Spearman, estimated vs true\nper-cell contamination"),
        ]:
            for col in cols:
                line, = ax.plot(sim_main["n_labels"], sim_main[col], "-", lw=1, label=col.replace("_", " "))
                for _, row in sim_main.iterrows():
                    ax.scatter(row["n_labels"], row[col], marker=styles[row["condition"].split("_")[0]], s=18, color=line.get_color(), zorder=3)
                ax.scatter(shuf["n_labels"], shuf[col], marker="x", color="#C44E52", s=30, zorder=4)
            ax.axvline(12, color="#2CA02C", lw=1, ls=":")
            ax.set_xscale("log")
            ax.set_xlabel("Number of labels (true K = 12)")
            ax.set_ylabel(ylab)
            ax.legend(fontsize=6, frameon=False)
        fig.suptitle("Simulation with ground truth: merged types (squares), truth (star), random splits (triangles), Leiden (circles), shuffled (red x)", fontsize=7.5)
        fig.tight_layout()
        for ext in ["png", "pdf"]:
            fig.savefig(os.path.join(gran_out_dir, f"celltype_granularity_simulation.{ext}"), dpi=300)
        plt.close(fig)

In [7]:
def main():
    ref = labels["ct_low"].map(REF_GROUP)
    assert ref.notna().all(), labels.loc[ref.isna(), "ct_low"].unique()
    ref = ref.values
    order = condition_order()

    summary, lineage_rows, subtype_rows = [], [], []
    per_cell_frac, per_gene_frac = {}, {}
    hvg_idx = None
    raw_embed = None

    for cond in order:
        print(f"scoring {cond}", flush=True)
        a, raw, den = load(cond, None)
        var_names = a.var_names
        removed = raw - den
        removed.data = np.clip(removed.data, 0, None)

        if hvg_idx is None:
            # fixed HVG set from the raw cells, shared by every run
            tmp = ad.AnnData(X=raw.copy(), var=pd.DataFrame(index=var_names))
            sc.pp.highly_variable_genes(tmp, n_top_genes=2000, flavor="seurat_v3")
            hvg_idx = np.where(tmp.var["highly_variable"].values)[0]
            raw_embed = embed(raw, hvg_idx)
            raw_ref_purity = knn_purity(raw_embed, ref)

        raw_cell = np.asarray(raw.sum(1)).ravel()
        per_cell_frac[cond] = np.asarray(removed.sum(1)).ravel() / raw_cell
        raw_gene = np.asarray(raw.sum(0)).ravel()
        keep_gene = raw_gene >= 100
        per_gene_frac[cond] = np.asarray(removed.sum(0)).ravel()[keep_gene] / raw_gene[keep_gene]

        rawc, denc = raw.tocsc(), den.tocsc()
        gidx = {g: i for i, g in enumerate(var_names)}

        # 1. lineage-marker specificity
        off_raw_all = off_rem_all = on_raw_all = on_rem_all = 0.0
        for panel, (genes, on, off) in LINEAGE_PANELS.items():
            cols = [gidx[g] for g in genes]
            r, d = rawc[:, cols], denc[:, cols]
            on_mask, off_mask = np.isin(ref, list(on)), np.isin(ref, list(off))
            off_raw, off_rem = r[off_mask].sum(), (r[off_mask] - d[off_mask]).sum()
            on_raw, on_rem = r[on_mask].sum(), (r[on_mask] - d[on_mask]).sum()
            lineage_rows.append(dict(condition=cond, panel=panel, offtarget_removed=off_rem / off_raw,
                                     ontarget_removed=(on_rem / on_raw) if on_raw > 0 else np.nan,
                                     offtarget_raw_counts=off_raw, ontarget_raw_counts=on_raw))
            if on_raw > 0:
                off_raw_all += off_raw; off_rem_all += off_rem; on_raw_all += on_raw; on_rem_all += on_rem

        # 2. subtype markers
        for g, (pos, neg) in SUBTYPE_MARKERS.items():
            c = gidx[g]
            r, d = np.asarray(rawc[:, c].todense()).ravel(), np.asarray(denc[:, c].todense()).ravel()
            pm, nm = ref == pos, ref == neg
            # pseudobulk CPM using each matrix's own library sizes
            den_cell = np.asarray(den.sum(1)).ravel()
            cpm = lambda x, lib, m: 1e6 * x[m].sum() / lib[m].sum()
            lfc_raw = np.log2((cpm(r, raw_cell, pm) + 1) / (cpm(r, raw_cell, nm) + 1))
            lfc_den = np.log2((cpm(d, den_cell, pm) + 1) / (cpm(d, den_cell, nm) + 1))
            subtype_rows.append(dict(condition=cond, gene=g, expressing=pos, sibling=neg,
                                     retained_in_expressing=d[pm].sum() / r[pm].sum(),
                                     removed_in_sibling=1 - d[nm].sum() / r[nm].sum(),
                                     log2fc_raw=lfc_raw, log2fc_denoised=lfc_den, log2fc_gain=lfc_den - lfc_raw))

        # 4. downstream structure
        emb = embed(den, hvg_idx)
        own = labels[cond].values
        summary.append(dict(
            condition=cond, label=pretty(cond), n_labels=labels[cond].nunique(),
            median_cells_per_label=float(labels[cond].value_counts().median()),
            min_cells_per_label=int(labels[cond].value_counts().min()),
            fraction_counts_removed=removed.sum() / raw.sum(),
            median_alpha_hat=float(np.median(a.obs["alpha_hat"])),
            n_cells_reassigned=int((np.asarray(a.uns["celltype_names"]).astype(str)[a.obs["z_hat"].values.astype(int) - 1] != own).sum()),  # z_hat is a 1-based index into celltype_names
            beta_hat=float(a.uns["beta_hat"]),
            loglike=float(a.uns["loglike"]),
            pooled_offtarget_marker_removed=off_rem_all / off_raw_all,
            pooled_ontarget_marker_removed=on_rem_all / on_raw_all,
            ref_knn_purity=knn_purity(emb, ref), ref_knn_purity_raw=raw_ref_purity,
            own_label_knn_purity=knn_purity(emb, own), own_label_knn_purity_raw=knn_purity(raw_embed, own),
        ))

    summary = pd.DataFrame(summary)
    for other in [DEFAULT, "ct_low"]:
        summary[f"cell_removed_frac_spearman_vs_{other}"] = [spearmanr(per_cell_frac[c], per_cell_frac[other]).correlation for c in summary.condition]
        summary[f"gene_removed_frac_spearman_vs_{other}"] = [spearmanr(per_gene_frac[c], per_gene_frac[other]).correlation for c in summary.condition]
    summary["marker_specificity"] = summary["pooled_offtarget_marker_removed"] - summary["pooled_ontarget_marker_removed"]
    summary["own_label_purity_gain"] = summary["own_label_knn_purity"] - summary["own_label_knn_purity_raw"]
    summary["ref_purity_gain"] = summary["ref_knn_purity"] - summary["ref_knn_purity_raw"]
    lineage = pd.DataFrame(lineage_rows)
    subtype = pd.DataFrame(subtype_rows)

    summary.to_csv(os.path.join(gran_out_dir, "summary_metrics.csv"), index=False)
    lineage.to_csv(os.path.join(gran_out_dir, "lineage_marker_metrics.csv"), index=False)
    subtype.to_csv(os.path.join(gran_out_dir, "subtype_marker_metrics.csv"), index=False)

    pd.set_option("display.width", 300)
    pd.set_option("display.max_columns", 50)
    print(summary.drop(columns=["label"]).round(3).to_string(index=False))
    print(lineage.pivot(index="panel", columns="condition", values="offtarget_removed")[order].round(3).to_string())
    print(lineage.pivot(index="panel", columns="condition", values="ontarget_removed")[order].round(3).to_string())
    print(subtype.pivot(index="gene", columns="condition", values="retained_in_expressing")[order].round(3).to_string())
    print(subtype.pivot(index="gene", columns="condition", values="log2fc_gain")[order].round(3).to_string())

    make_figure(summary, lineage, subtype, order)


main()

scoring all_one


/home/mcaskey/.conda/envs/cellsweep/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


/home/mcaskey/.conda/envs/cellsweep/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


scoring leiden_0.001


/home/mcaskey/.conda/envs/cellsweep/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


...

scoring leiden_20


/home/mcaskey/.conda/envs/cellsweep/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


scoring shuffled


/home/mcaskey/.conda/envs/cellsweep/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


   condition  n_labels  median_cells_per_label  min_cells_per_label  fraction_counts_removed  median_alpha_hat  n_cells_reassigned  beta_hat  loglike  pooled_offtarget_marker_removed  pooled_ontarget_marker_removed  ref_knn_purity  ref_knn_purity_raw  own_label_knn_purity  own_label_knn_purity_raw  cell_removed_frac_spearman_vs_ct_high  gene_removed_frac_spearman_vs_ct_high  cell_removed_frac_spearman_vs_ct_low  gene_removed_frac_spearman_vs_ct_low  marker_specificity  own_label_purity_gain  ref_purity_gain
     all_one         1                  8381.0                 8381                    0.474             0.235                   0     0.039 -418.846                            0.897                           0.894           0.933               0.943                 1.000                     1.000                                  0.436                                  0.243                                 0.322                                 0.248               0.003               

## 6. Signal and noise retained (Fig. S19)

Lineage-marker counts kept in expressing cells (signal) and in cells of other lineages, where they are ambient (noise),
against the number of labels (A) and of non-cellular barcodes (B).

In [5]:
matplotlib.rcdefaults()   # section 5 changes the font size and spines globally

matplotlib.use("Agg")
EMPTY_TAGS = ["10", "100", "1000", "10000", "50000", "100000", "500000", "all"]
N_EMPTY_ALL = 728899
EXAMPLES = {
    "CD16 monocyte program (FCGR3A, CDKN1C, MS4A7)": (["FCGR3A", "CDKN1C", "MS4A7"], {"CD16 Mono"}, T | B, "#4C72B0"),
    "NK program (NKG7, GNLY, KLRF1)": (["NKG7", "GNLY", "KLRF1"], {"NK"}, B | MYELOID, "#C44E52"),
}
def series(path, ref):
    """Fraction of marker counts retained, in expressing cells (signal) and in other cells (noise)."""
    a = ad.read_h5ad(path)
    a = a[~a.obs["is_empty"].values].copy()
    a.var_names_make_unique()
    a = a[labels.index]
    raw = sp.csr_matrix(a.layers["raw"], dtype=np.float64).tocsc()
    den = sp.csr_matrix(a.X, dtype=np.float64).tocsc()
    gidx = {g: i for i, g in enumerate(a.var_names)}

    out = {}
    sig_raw = sig_den = noi_raw = noi_den = 0.0
    for panel, (genes, on, off) in LINEAGE_PANELS.items():
        cols = [gidx[g] for g in genes]
        r, d = raw[:, cols], den[:, cols]
        on_mask, off_mask = np.isin(ref, list(on)), np.isin(ref, list(off))
        noi_raw += r[off_mask].sum(); noi_den += d[off_mask].sum()
        if on_mask.sum():
            sig_raw += r[on_mask].sum(); sig_den += d[on_mask].sum()
    out["signal_retained"] = sig_den / sig_raw
    out["noise_retained"] = noi_den / noi_raw

    for name, (genes, on, off, _color) in EXAMPLES.items():
        cols = [gidx[g] for g in genes]
        r, d = raw[:, cols], den[:, cols]
        on_mask, off_mask = np.isin(ref, list(on)), np.isin(ref, list(off))
        out[f"{name}|signal_retained"] = d[on_mask].sum() / r[on_mask].sum()
        out[f"{name}|noise_retained"] = d[off_mask].sum() / r[off_mask].sum()
    return out
def collect():
    ref = labels["ct_low"].map(REF_GROUP).values
    rows = []
    for cond in labels.columns:
        print(f"scoring labels: {cond}", flush=True)
        row = dict(sweep="labels", condition=cond, x=labels[cond].nunique())
        row.update(series(os.path.join(run_dir, f"adata_cellsweep_{cond}.h5ad"), ref))
        rows.append(row)
    for tag in EMPTY_TAGS:
        print(f"scoring empty barcodes: {tag}", flush=True)
        row = dict(sweep="empty", condition=tag, x=N_EMPTY_ALL if tag == "all" else int(tag))
        row.update(series(os.path.join(empty_run_dir, f"adata_cellsweep_empty_{tag}.h5ad"), ref))
        rows.append(row)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(gran_out_dir, "signal_vs_noise_retained.csv"), index=False)
    return df
def draw(ax, df, color="k", label_prefix="", sig_col="signal_retained", noi_col="noise_retained"):
    df = df.sort_values("x")
    ax.plot(df["x"], df[sig_col], "-", color=color, lw=1.2, marker="o", ms=5, mfc=color, mec=color, label=f"{label_prefix}signal retained")
    ax.plot(df["x"], df[noi_col], "--", color=color, lw=1.2, marker="o", ms=5, mfc="none", mec=color, label=f"{label_prefix}noise retained")
def finish(ax, xlabel):
    ax.set_xscale("log")
    ax.set_ylim(-0.03, 1.03)
    ax.set_yticks(np.arange(0, 1.01, 0.1))
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Fraction of counts retained")
    ax.axhline(1, color="0.85", lw=0.8, zorder=0)
    ax.axhline(0, color="0.85", lw=0.8, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
def save_fig(fig, name, tight_bbox=False):
    for ext in ["png", "pdf"]:
        fig.savefig(os.path.join(gran_out_dir, f"{name}.{ext}"), dpi=300, bbox_inches="tight" if tight_bbox else None)
    plt.close(fig)
def save_legend(handles, labels, name):
    """Legend as its own file, one entry per line."""
    fig = plt.figure(figsize=(3.4, 0.32 * len(labels) + 0.2))
    fig.legend(handles, labels, loc="center", frameon=False, fontsize=8, ncol=1)
    save_fig(fig, name, tight_bbox=True)
def figure_labels(df):
    d = df[df.sweep == "labels"].set_index("condition")
    # Leiden sweep; a single label for every cell (all_one) is its K = 1 limit
    leiden = d.loc[["all_one"] + [c for c in d.index if c.startswith("leiden")]]
    fig, ax = plt.subplots(figsize=(5.4, 4.2))
    draw(ax, leiden)

    # CellTypist annotations, plotted as squares (High) and diamonds (Low)
    red = "#d62728"
    ct_handles = []
    for cond, marker, name in [("ct_high", "s", "CellTypist High"), ("ct_low", "D", "CellTypist Low")]:
        row = d.loc[cond]
        filled = ax.scatter(row["x"], row["signal_retained"], marker=marker, s=55, color=red, zorder=5)
        hollow = ax.scatter(row["x"], row["noise_retained"], marker=marker, s=55, facecolor="none", edgecolor=red, lw=1.2, zorder=5)
        ax.annotate(name, (row["x"], row["noise_retained"]), textcoords="offset points", xytext=(0, -13), ha="center", fontsize=6.5, color=red)
        ct_handles.append(((filled, hollow), name))

    finish(ax, "Number of celltypes (Leiden)")
    handles, labels_ = ax.get_legend_handles_labels()
    handles += [h for h, _ in ct_handles]
    labels_ += [n for _, n in ct_handles]
    ax.legend(handles, labels_, handler_map={tuple: HandlerTuple(ndivide=None, pad=0.3)}, loc="lower right",
              frameon=False, fontsize=8)
    fig.tight_layout()
    save_fig(fig, "celltype_granularity_signal_vs_noise")
def figure_empty(df):
    d = df[df.sweep == "empty"]
    fig, ax = plt.subplots(figsize=(5.4, 4.2))
    draw(ax, d)
    ax.axvline(N_EMPTY_ALL, color="0.6", ls=":", lw=1)
    ax.annotate(f"all {N_EMPTY_ALL:,}", (N_EMPTY_ALL, 0.5), rotation=90, va="center", ha="right", fontsize=6.5, color="0.35")
    finish(ax, "Number of empty barcodes")
    ax.legend(loc="center right", bbox_to_anchor=(0.97, 0.25), frameon=False, fontsize=8)
    fig.tight_layout()
    save_fig(fig, "empty_barcode_signal_vs_noise")

df = collect()
figure_labels(df)
figure_empty(df)
df.round(3)

scoring labels: ct_high


scoring labels: all_one


scoring labels: lineage


scoring labels: shuffled


scoring labels: ct_low


...

scoring empty barcodes: 50000


scoring empty barcodes: 100000


scoring empty barcodes: 500000


scoring empty barcodes: all


,sweep,condition,x,signal_retained,noise_retained,"CD16 monocyte program (FCGR3A, CDKN1C, MS4A7)|signal_retained","CD16 monocyte program (FCGR3A, CDKN1C, MS4A7)|noise_retained","NK program (NKG7, GNLY, KLRF1)|signal_retained","NK program (NKG7, GNLY, KLRF1)|noise_retained"
0,labels,ct_high,7,0.936,0.045,0.879,0.226,0.979,0.051
1,labels,all_one,1,0.106,0.103,0.000,0.000,0.089,0.116
2,labels,lineage,2,0.890,0.106,0.881,0.001,0.095,0.118
3,labels,shuffled,7,0.904,0.132,0.813,0.156,0.787,0.253
4,labels,ct_low,18,0.941,0.090,0.991,0.516,0.983,0.084
5,labels,leiden_0.001,2,0.890,0.107,0.881,0.002,0.096,0.119
6,labels,leiden_0.005,3,0.907,0.031,0.878,0.195,0.263,0.004
7,labels,leiden_0.01,3,0.907,0.033,0.878,0.192,0.257,0.004
8,labels,leiden_0.1,6,0.938,0.047,0.878,0.514,0.989,0.035
9,labels,leiden_0.5,14,0.941,0.090,0.984,0.532,0.979,0.144
